# Transformer Cross-Encoders for Stance Detection on args.me

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

In the previous stage, traditional rule-based classifiers and shallow NLP models (TF-IDF + Logistic Regression / SVM) hit a **performance ceiling of ~55% accuracy**, heavily distorted by sentiment mismatch and adversarial quoting (e.g. CON debaters repeatedly quoting `"pro"`).

This notebook demonstrates how **Transformer Cross-Encoders** (such as `microsoft/deberta-v3-small`) overcome these barriers by processing paired representations:

$$\text{Input: } [\text{CLS}] \; \text{Conclusion/Topic} \; [\text{SEP}] \; \text{Premise} \; [\text{SEP}]$$

Through bidirectional cross-attention, every token in the premise directly conditions on the debate motion, enabling the model to learn true **relational stance entailment**.

In [ ]:
# 1. Environment Detection and Dependency Setup
import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if IN_COLAB:
    print("Provisioning dependencies inside Google Colab runtime...")
    !pip install -q datasets huggingface_hub pandas pyarrow scikit-learn torch transformers accelerate sentencepiece tiktoken matplotlib seaborn
else:
    print("Running in local environment.")

In [ ]:
# 2. Imports and Hardware Acceleration Check
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Ensure project root in sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.transformer_stance import (
    TransformerStanceCrossEncoder,
    ZeroShotNLIStanceClassifier,
    get_compute_device,
)

device = get_compute_device()
print(f"Active PyTorch Compute Device: {device}")
if device.type == "cuda":
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
elif device.type == "mps":
    print("Using Apple Silicon Metal Performance Shaders (MPS GPU).")
else:
    print("Using CPU runtime.")

In [ ]:
# 3. Load Benchmark Dataset
data_path = Path("../data/args_me_benchmark_5k.parquet")
if not data_path.exists():
    data_path = Path("data/args_me_benchmark_5k.parquet")

df = pd.read_parquet(data_path)
df_clean = df[df["stance"].isin(["PRO", "CON"])].copy().reset_index(drop=True)

# Split into Train (72%), Val (8%), and Test (20%)
train_val_df, test_df = train_test_split(
    df_clean, test_size=0.20, random_state=42, stratify=df_clean["stance"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.10, random_state=42, stratify=train_val_df["stance"]
)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")
print(f"Test stance distribution:\n{test_df['stance'].value_counts()}")

In [ ]:
# 4. Baseline: Zero-Shot Out-of-the-Box NLI Cross-Encoder
print("Evaluating Zero-Shot NLI Cross-Encoder (cross-encoder/nli-deberta-v3-small)...")
zero_shot_model = ZeroShotNLIStanceClassifier(
    model_name="cross-encoder/nli-deberta-v3-small",
    device=device,
    batch_size=16,
)
zs_results = zero_shot_model.evaluate_df(test_df)
print(f"Zero-Shot NLI Accuracy: {zs_results.accuracy:.4f}")
print(f"Zero-Shot Macro F1:    {zs_results.macro_f1:.4f}")
print(f"Zero-Shot ROC-AUC:     {zs_results.roc_auc:.4f}")

In [ ]:
# 5. Fine-Tune DeBERTa-v3-small Cross-Encoder
cross_encoder = TransformerStanceCrossEncoder(
    model_name="microsoft/deberta-v3-small",
    max_length=160,
    batch_size=16,
    learning_rate=2e-5,
    device=device,
)

# Execute fine-tuning
train_history = cross_encoder.fit(
    train_df=train_df,
    val_df=val_df,
    epochs=2,
    warmup_ratio=0.1,
)

In [ ]:
# 6. Evaluate Fine-Tuned Cross-Encoder on Test Set
ft_results = cross_encoder.evaluate_df(test_df)
print("=== FINE-TUNED DEBERTA-V3 BENCHMARK RESULTS ===")
print(f"Accuracy:         {ft_results.accuracy:.4f}")
print(f"Macro F1:         {ft_results.macro_f1:.4f}")
print(f"CON F1:           {ft_results.con_f1:.4f} | PRO F1: {ft_results.pro_f1:.4f}")
print(f"ROC-AUC:          {ft_results.roc_auc:.4f}")
print(f"Confusion Matrix: {ft_results.confusion_mat}")

In [ ]:
# 7. Comprehensive Paradigm Comparison: Traditional vs. Deep Cross-Encoder
comparison_data = [
    {"Model": "Majority Baseline (Zero-R)", "Accuracy": 0.5020, "Macro F1": 0.3342, "Type": "Baseline"},
    {"Model": "Rule-Based Heuristic Matcher", "Accuracy": 0.4920, "Macro F1": 0.4902, "Type": "Rule-Based"},
    {"Model": "VADER Sentiment Polarity", "Accuracy": 0.5040, "Macro F1": 0.4988, "Type": "Lexicon"},
    {"Model": "Random Forest (Linguistic Features)", "Accuracy": 0.5120, "Macro F1": 0.5117, "Type": "Traditional ML"},
    {"Model": "Linear SVM (Contextual TF-IDF)", "Accuracy": 0.5070, "Macro F1": 0.5067, "Type": "Traditional ML"},
    {"Model": "Logistic Regression (Premise TF-IDF)", "Accuracy": 0.5490, "Macro F1": 0.5490, "Type": "Traditional ML"},
    {"Model": "Zero-Shot NLI Cross-Encoder", "Accuracy": zs_results.accuracy, "Macro F1": zs_results.macro_f1, "Type": "Transformer"},
    {"Model": "Fine-Tuned DeBERTa-v3 Cross-Encoder", "Accuracy": ft_results.accuracy, "Macro F1": ft_results.macro_f1, "Type": "Transformer"},
]
comp_df = pd.DataFrame(comparison_data).sort_values("Accuracy", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#94a3b8" if t in ["Baseline", "Rule-Based", "Lexicon"] else ("#3b82f6" if t == "Traditional ML" else "#10b981") for t in comp_df["Type"]]
bars = ax.barh(comp_df["Model"], comp_df["Accuracy"], color=colors, height=0.6)
ax.axvline(0.50, color="#ef4444", linestyle="--", linewidth=1.5, label="Chance Baseline (50%)")

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height()/2, f"{w:.1%}", va="center", fontsize=11, fontweight="bold")

ax.set_xlim(0.40, 0.85)
ax.set_xlabel("Test Set Accuracy", fontsize=12)
ax.set_title("Stance Classification Performance Across Architectural Eras", fontsize=14, fontweight="bold", pad=12)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Qualitative Error Analysis: Handling Adversarial Quotes & Negations
sample_cases = test_df.sample(5, random_state=42)[["conclusion", "premise_text", "stance"]]
preds, probs = cross_encoder.predict(sample_cases["conclusion"].tolist(), sample_cases["premise_text"].tolist())
sample_cases["Prediction"] = preds
sample_cases["PRO Probability"] = probs.round(4)
sample_cases["Correct"] = sample_cases["stance"] == sample_cases["Prediction"]
sample_cases